# AutoML · M06: TabPFN v2

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M06 — TabPFN v2 |

---

## 🎯 Qué hace

Evalúa TabPFN v2 (Hollmann et al., 2024) sobre el dataset completo de
abandono universitario. TabPFN v2 es un transformer preentrenado en
millones de datasets sintéticos que realiza inferencia bayesiana aproximada
sin entrenamiento ni ajuste de hiperparámetros.

A diferencia de TabPFN v1, la versión 2 escala a datasets grandes sin
límite de muestras ni features, lo que permite evaluarlo sobre el dataset
completo con el mismo split 70/30 que el resto de módulos.

Incluido a propuesta del tutor Raúl Parada como diferenciador metodológico
de nivel cum laude — representa el estado del arte en clasificación tabular.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- Entorno: `tfm_abandono`
- Librería: `tabpfn` (`pip install tabpfn`)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_tabpfn.parquet` | Métricas de TabPFN v2 |
| `docs/html/fase_automl/m06_tabpfn.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ Split 70/30 estratificado (igual que M01-M05)
    ↓ TabPFNClassifier v2 — sin entrenamiento, inferencia directa
    ↓ Métricas: F1, AUC, Precision, Recall, MCC (umbral 0.5)
    → results_tabpfn.parquet + HTML
```

## ➡️ Siguiente

`fautoml_m07_comparativa.ipynb` — ranking global de todos los frameworks

---

> **Nota metodológica:** TabPFN v2 no requiere hiperparámetros ni
> entrenamiento explícito — el modelo viene preentrenado. El 'fit' solo
> adapta el contexto al dataset. Resultados directamente comparables
> con M01-M05 al usar el mismo split 70/30 y umbral 0.5.


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import time
import subprocess
from pathlib import Path

warnings.filterwarnings('ignore')

# Verificar e instalar TabPFN si falta
try:
    import tabpfn
    print(f'✅ TabPFN {tabpfn.__version__} disponible')
except ImportError:
    print('⚠️  Instalando tabpfn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'tabpfn'])
    import tabpfn
    print(f'✅ TabPFN instalado')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tabpfn import TabPFNClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss
)

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
FRAMEWORK     = 'tabpfn'
NOTEBOOK_NAME = 'fautoml_m06_tabpfn.ipynb'
CARPETA_NB    = 'fase_automl'

info_entorno()
print('\n✅ Configuración completada')


✅ TabPFN 7.1.1 disponible
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proy

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET     = 'abandono'
n_total    = len(df)
n_features = len(df.columns) - 1
n_abandono = int(df[TARGET].sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset : {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')

# Verificación anti-leakage
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in df.columns]
if leakage_presente:
    raise ValueError(f'LEAKAGE DETECTADO: {leakage_presente}')
print('\n✅ Verificación anti-leakage superada')

# Split 70/30 estratificado — igual que M01-M05
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.30,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f'\nSplit 70/30 estratificado:')
print(f'  Train : {fmt(len(X_train))} ({(y_train==1).mean()*100:.1f}% abandono)')
print(f'  Test  : {fmt(len(X_test))}  ({(y_test==1).mean()*100:.1f}% abandono)')

print(f'\n📋 Features ({n_features}):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO

✓ Dataset : dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)

✅ Verificación anti-leakage superada

Split 70/30 estratificado:
  Train : 23.534 (29.2% abandono)
  Test  : 10.087  (29.2% abandono)

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: TABPFN V2 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
# Usa tabpfn local con ignore_pretraining_limits=True para dataset completo.
# Si va lento, alternativa: tabpfn_client (API en la nube sin límites).
# Métricas con sklearn umbral 0.5 — comparables con M01-M05.
# ============================================================================

import os
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

print('\n' + '='*70)
print('TABPFN V2 — INFERENCIA SOBRE DATASET COMPLETO')
print('='*70)
print(f'\n🧠 Iniciando TabPFN v2 (CPU, dataset completo)...')
print(f'   ⏱️  Puede tardar varios minutos en CPU...')

t0 = time.time()

clf = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
clf.fit(X_train, y_train)

y_prob_raw = clf.predict_proba(X_test)
# predict_proba puede devolver DataFrame o array
if hasattr(y_prob_raw, 'values'):
    y_prob = y_prob_raw.values[:, 1]
else:
    y_prob = y_prob_raw[:, 1]

y_pred  = (y_prob >= 0.5).astype(int)
t_total = round(time.time() - t0, 2)

# Métricas con sklearn umbral 0.5
metricas = {
    'framework'        : FRAMEWORK,
    'model_name'       : 'TabPFN v2',
    'accuracy'         : accuracy_score(y_test, y_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
    'precision'        : precision_score(y_test, y_pred, zero_division=0),
    'recall'           : recall_score(y_test, y_pred, zero_division=0),
    'f1'               : f1_score(y_test, y_pred, zero_division=0),
    'auc_roc'          : roc_auc_score(y_test, y_prob),
    'mcc'              : matthews_corrcoef(y_test, y_pred),
    'kappa'            : cohen_kappa_score(y_test, y_pred),
    'log_loss'         : round(log_loss(y_test, y_prob), 4),
    'train_time_sec'   : t_total,
}

df_resultados = pd.DataFrame([metricas])

print(f'\n✅ TabPFN v2 completado en {t_total}s')
print(f'\n--- RESULTADOS ---')
for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc', 'kappa', 'log_loss']:
    print(f'  {campo:<20}: {metricas[campo]:.4f}')



TABPFN V2 — INFERENCIA SOBRE DATASET COMPLETO

🧠 Iniciando TabPFN v2 (CPU, dataset completo)...
   ⏱️  Puede tardar varios minutos en CPU...

✅ TabPFN v2 completado en 26230.84s

--- RESULTADOS ---
  f1                  : 0.8496
  auc_roc             : 0.9644
  precision           : 0.8946
  recall              : 0.8088
  mcc                 : 0.7937
  kappa               : 0.7917
  log_loss            : 0.2091


In [4]:
# ============================================================================
# CELDA 4: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_tabpfn.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: guardado')


💾 results_tabpfn.parquet: guardado


In [5]:
# ============================================================================
# CELDA 5: GRÁFICOS Y HTML
# ============================================================================

graficos_b64 = {}

# --- Gráfico: comparativa TabPFN vs otros frameworks ---
# Cargar resultados de módulos anteriores para comparativa visual
frameworks_data = []
for fname, flabel in [
    ('results_baselines.parquet',   'M01 Baselines'),
    ('results_pycaret.parquet',     'M03 PyCaret'),
    ('results_h2o.parquet',         'M04 H2O'),
    ('results_autogluon.parquet',   'M05 AutoGluon'),
]:
    ruta_fw = RUTA_AUTOML / fname
    if ruta_fw.exists():
        df_fw    = pd.read_parquet(ruta_fw)
        no_dummy = ~df_fw['model_name'].str.startswith('Dummy')
        mejor_fw = df_fw[no_dummy].sort_values('f1', ascending=False).iloc[0]
        frameworks_data.append({
            'label': flabel,
            'f1'   : mejor_fw['f1'],
            'auc'  : mejor_fw['auc_roc'],
            'mcc'  : mejor_fw['mcc'],
        })

# Añadir TabPFN
frameworks_data.append({
    'label': 'M06 TabPFN v2',
    'f1'   : metricas['f1'],
    'auc'  : metricas['auc_roc'],
    'mcc'  : metricas['mcc'],
})

if frameworks_data:
    labels = [d['label'] for d in frameworks_data]
    f1s    = [d['f1']    for d in frameworks_data]
    aucs   = [d['auc']   for d in frameworks_data]

    fig, ax = plt.subplots(figsize=(12, 5))
    x     = np.arange(len(labels))
    width = 0.35

    bars_f1  = ax.bar(x - width/2, f1s,  width, label='F1',     color='#3182ce', alpha=0.85)
    bars_auc = ax.bar(x + width/2, aucs, width, label='AUC-ROC', color='#e53e3e', alpha=0.85)

    # Destacar TabPFN
    bars_f1[-1].set_edgecolor('black')
    bars_f1[-1].set_linewidth(2)
    bars_auc[-1].set_edgecolor('black')
    bars_auc[-1].set_linewidth(2)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=10)
    ax.set_ylabel('Score')
    ax.set_title('TabPFN v2 vs otros frameworks AutoML', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_ylim(0.5, 1.0)
    ax.axhline(y=0.83, color='gray', ls='--', alpha=0.3)

    for bar in bars_f1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    graficos_b64['comparativa'] = figura_a_base64(fig)
    plt.close()

# --- KPIs ---
KPIS = [
    {'valor': fmt(n_total),                'titulo': 'Expedientes'},
    {'valor': str(n_features),             'titulo': 'Features'},
    {'valor': f'{metricas["f1"]:.4f}',     'titulo': 'F1 (umbral 0.5)'},
    {'valor': f'{metricas["auc_roc"]:.4f}','titulo': 'AUC-ROC'},
]

# --- Sección metodología ---
s_metod = generar_seccion_html('Metodología — TabPFN v2', f'''
<p><strong>TabPFN v2</strong> (Hollmann et al., 2024) es un transformer
preentrenado en millones de datasets sintéticos. Realiza inferencia bayesiana
aproximada sin entrenamiento ni ajuste de hiperparámetros — el modelo ya viene
listo. La versión 2 elimina las limitaciones de tamaño de la v1 y escala
a datasets grandes.</p>
<div style="background:#ebf8ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #3182ce;">
  <strong>Por qué está aquí:</strong> Propuesta del tutor Raúl Parada.
  Representa el estado del arte en clasificación tabular sin hiperparámetros.
  Sus resultados son directamente comparables con M01-M05 al usar el mismo
  split 70/30 estratificado y umbral 0.5.
</div>
<div style="background:#f0fff4;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #38a169;">
  <strong>Ventaja metodológica:</strong> TabPFN v2 no requiere ningún
  ajuste — es un baseline de referencia "zero-shot" que muestra cuánto
  conocimiento transferible existe en la estructura tabular del problema.
</div>''', 'ℹ️')

# --- Tabla de resultados ---
tabla = '<table style="width:100%;border-collapse:collapse;font-size:13px;">\n'
tabla += '<tr style="background:#3182ce;color:white;">'
for col in ['Modelo', 'F1', 'AUC', 'Precision', 'Recall', 'MCC', 'Kappa', 'LogLoss', 'Tiempo']:
    tabla += f'<th style="padding:10px;text-align:center;">{col}</th>'
tabla += '</tr>\n'
row = df_resultados.iloc[0]
tabla += '<tr style="background:#ebf8ff;font-weight:bold;">'
tabla += f'<td style="padding:8px;">TabPFN v2</td>'
for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc', 'kappa', 'log_loss']:
    v = row[campo]
    color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
    tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
tabla += f'<td style="text-align:center;">{row["train_time_sec"]:.1f}s</td>'
tabla += '</tr>\n</table>'
s_tabla = generar_seccion_html('Resultados TabPFN v2 (dataset completo, umbral 0.5)', tabla, '📊')

# --- Gráfico comparativa ---
s_grafico = ''
if 'comparativa' in graficos_b64:
    graf = f'<img src="data:image/png;base64,{graficos_b64["comparativa"]}" style="max-width:100%;border-radius:8px;">'
    s_grafico = generar_seccion_html('TabPFN v2 vs otros frameworks', graf, '📈')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm06_tabpfn.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_metod + s_tabla + s_grafico,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m06_tabpfn.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN FINAL
# ============================================================================

print('\n' + '='*60)
print('✅ AUTOML-M06 COMPLETADO')
print('='*60)
print(f'Framework     : TabPFN v2 (Hollmann et al., 2024)')
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} x {n_features+1})')
print(f'Split         : 70/30 estratificado (igual que M01-M05)')
print(f'Umbral        : 0.5 fijo')
print(f'F1            : {metricas["f1"]:.4f}')
print(f'AUC           : {metricas["auc_roc"]:.4f}')
print(f'MCC           : {metricas["mcc"]:.4f}')
print(f'Tiempo        : {metricas["train_time_sec"]:.1f}s')
print(f'Resultados    : {RUTA_AUTOML / "results_tabpfn.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m06_tabpfn.html"}')
print(f'\n➡️  Siguiente  : fautoml_m07_comparativa.ipynb')
print('='*60)



✅ AUTOML-M06 COMPLETADO
Framework     : TabPFN v2 (Hollmann et al., 2024)
Dataset       : dataset_final_tfm.parquet (33.621 x 25)
Split         : 70/30 estratificado (igual que M01-M05)
Umbral        : 0.5 fijo
F1            : 0.8496
AUC           : 0.9644
MCC           : 0.7937
Tiempo        : 26230.8s
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_tabpfn.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m06_tabpfn.html

➡️  Siguiente  : fautoml_m07_comparativa.ipynb
